# Tune a model with Optuna

Hyperparameter tuning can substantially improve a model's performance. To
make it easier to get started, sgtlearn provides a recommended search space
through `suggest_params`, a helper that works directly with Optuna trials.

This tutorial shows how to use it to tune a classifier, make predictions,
and apply the same workflow to regression.

To get started, install the tuning extra in your notebook environment:

```console
pip install 'sgtlearn[tune]'
```

We'll use two small datasets included with scikit-learn, so there's nothing
else to download.

In [1]:
import optuna
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.metrics import balanced_accuracy_score, mean_squared_error
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, train_test_split

from sgtlearn import RandomSGForestClassifier, SGTClassifier, SGTRegressor
from sgtlearn.tune import suggest_params

optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42
N_TRIALS = 10

## 1. Train your first classifier

SGT estimators follow the scikit-learn API: create a model, call `fit`, then
call `predict`. You can start with the defaults and tune later.

Let's split the breast cancer dataset into training and test sets. We'll use
the training set to choose parameters and keep the test set for evaluation.

In [2]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED,
)

baseline = SGTClassifier(random_state=SEED).fit(X_train, y_train)
baseline.predict(X_train[:5])

array([0, 1, 0, 0, 0])

## 2. Choose whether to use TAO and bivariate splits

Ordinary SGT fits use ShapeCART with both options off:

- **TAO** refines the tree's routing rules after training.
- **Bivariate branching** lets a split use two features together to capture interactions.

Pass `use_tao` and `use_bivariate` to `suggest_params` to enable either during
tuning. Here we'll try both. Set either to `False` if you want to leave it off.
Both arguments default to `False`.

With TAO enabled, the helper uses one refinement sweep with `tao_lambda=1e-4`.
With bivariate branching enabled, it searches the pair-candidate budget and
pair penalty along with the other tree parameters.

In [3]:
options = {"use_tao": True, "use_bivariate": True}

## 3. Tell Optuna how to score a model

`suggest_params(trial)` returns a dictionary you can pass straight to an
estimator. Optuna calls the objective repeatedly, trying different dictionaries.

We'll score each classifier with three-fold cross-validation and return its
mean balanced accuracy. This gives each class equal weight, even when their
sample counts differ. Higher is better, so we'll maximize this score.

In [4]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)


def objective(trial):
    params = suggest_params(trial, **options)
    model = SGTClassifier(**params, random_state=SEED)
    scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring="balanced_accuracy",
    )
    return scores.mean()


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=N_TRIALS)
print(f"Best cross-validation balanced accuracy: {study.best_value:.3f}")

Best cross-validation balanced accuracy: 0.950


Ten trials keep this example short. Increase `N_TRIALS` to explore more
configurations on your own data.

The helper searches settings such as tree depth, inner leaf count, minimum
leaf sizes, and impurity penalties.

## 4. Fit the best model and make predictions

Rebuild the winning dictionary with `FixedTrial`, using the same options as
the search. This includes fixed settings that aren't stored in
`study.best_params`, such as the TAO controls.

Now fit on all the training data and compare with the default model on the
test set. A better cross-validation score doesn't guarantee a better test score.

In [5]:
params = suggest_params(optuna.trial.FixedTrial(study.best_trial.params), **options)
model = SGTClassifier(**params, random_state=SEED).fit(X_train, y_train)
predictions = model.predict(X_test)

print(f"Default model balanced accuracy: {balanced_accuracy_score(y_test, baseline.predict(X_test)):.3f}")
print(f"Tuned model balanced accuracy:   {balanced_accuracy_score(y_test, predictions):.3f}")
print("Predicted classes:", predictions[:5])
print("Class probabilities:")
print(model.predict_proba(X_test[:5]).round(3))

Default model balanced accuracy: 0.929
Tuned model balanced accuracy:   0.931
Predicted classes: [1 0 0 1 1]
Class probabilities:
[[0.    1.   ]
 [1.    0.   ]
 [1.    0.   ]
 [0.    1.   ]
 [0.294 0.706]]


The tuned model here also enables TAO and bivariate branching, so the
comparison reflects those choices as well as parameter tuning. To tune a
plain ShapeCART model instead, set both options to `False` and run a new study.

## 5. Use the same approach for regression

For a continuous target, use `SGTRegressor`. Here we'll predict the diabetes
dataset's progression score and minimize mean squared error (MSE).

Scikit-learn's `neg_mean_squared_error` scorer returns negative MSE. Negate
that score in the objective and set Optuna's direction to `"minimize"`.
The regressor handles target scaling internally; pass `y` as-is and predictions
come back in its original units.

In [6]:
X_reg, y_reg = load_diabetes(return_X_y=True)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=SEED,
)
regression_cv = KFold(n_splits=3, shuffle=True, random_state=SEED)


def regression_objective(trial):
    reg_params = suggest_params(trial, **options)
    regressor = SGTRegressor(**reg_params, random_state=SEED)
    scores = cross_val_score(
        regressor, Xr_train, yr_train, cv=regression_cv,
        scoring="neg_mean_squared_error",
    )
    return -scores.mean()


regression_study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
regression_study.optimize(regression_objective, n_trials=N_TRIALS)

reg_params = suggest_params(
    optuna.trial.FixedTrial(regression_study.best_trial.params), **options,
)
regressor = SGTRegressor(**reg_params, random_state=SEED).fit(Xr_train, yr_train)
reg_predictions = regressor.predict(Xr_test)

print(f"Best cross-validation MSE: {regression_study.best_value:.1f}")
print(f"Test MSE:                 {mean_squared_error(yr_test, reg_predictions):.1f}")
print("Predictions:", reg_predictions[:5].round(1))

Best cross-validation MSE: 4161.0
Test MSE:                 3208.1
Predictions: [149.8 222.7 149.8 222.7 139. ]


## 6. Try a forest

The suggested parameters also work with the random forest estimators.
Pass them to `RandomSGForestClassifier` or `RandomSGForestRegressor`, then
choose forest settings such as `n_estimators` separately.

This example uses 20 trees to keep fitting quick; the default is 100.
Treat parameters chosen for a single tree as a starting point for a forest,
and check the forest's performance for your task.

In [7]:
forest = RandomSGForestClassifier(
    **params, n_estimators=20, random_state=SEED,
).fit(X_train, y_train)
forest.predict(X_test[:5])

array([1, 0, 1, 1, 1])

## Try it with your data

Replace the dataset-loading cells with your own `X` and `y`, choose a scoring
metric, and increase the trial budget. Keep preprocessing inside the
cross-validation folds, for example with a scikit-learn `Pipeline`.
Use the test set to evaluate your final choice rather than to guide the search.

Keep `use_tao` and `use_bivariate` unchanged within a study and when rebuilding
its best parameters. If you want to compare different options, give each
combination its own study.

For more on the optional features, see [bivariate branching](bivariate-branching.ipynb)
and [TAO refinement](tao.ipynb).